In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torchvision.utils import save_image
import itertools
import os


In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residual_blocks=9):
        super(Generator, self).__init__()
        # Initial Convolution
        model = [
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=1, padding=3),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        ]

        # Downsampling for 1024x1024
        model += [
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 512x512
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # 256x256
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),  # 128x128
            nn.InstanceNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1024, kernel_size=3, stride=2, padding=1),  # 64x64
            nn.InstanceNorm2d(1024),
            nn.ReLU(inplace=True),
        ]

        # Residual Blocks
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(1024)]

        # Upsampling back to 1024x1024
        model += [
            nn.ConvTranspose2d(1024, 512, kernel_size=3, stride=2, padding=1, output_padding=1),  # 128x128
            nn.InstanceNorm2d(512),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),  # 256x256
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),  # 512x512
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # 1024x1024
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        ]

        # Final Output
        model += [nn.Conv2d(64, out_channels, kernel_size=7, stride=1, padding=3), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


In [4]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super(Discriminator, self).__init__()
        model = [
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid(),
        ]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


In [9]:
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [18]:
dataset_A = datasets.ImageFolder("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Image_correction_ink_marker", transform=transform)
dataset_B = datasets.ImageFolder("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Image_correction_ink_marker", transform=transform)

dataloader_A = DataLoader(dataset_A, batch_size=1, shuffle=True)
dataloader_B = DataLoader(dataset_B, batch_size=1, shuffle=True)


In [19]:
next(iter(dataloader_A))

[tensor([[[[0.5294, 0.5608, 0.5922,  ..., 0.6314, 0.6863, 0.6941],
           [0.4980, 0.5137, 0.5373,  ..., 0.6784, 0.7412, 0.7804],
           [0.4824, 0.4824, 0.4902,  ..., 0.7020, 0.7255, 0.8275],
           ...,
           [0.5059, 0.5059, 0.5059,  ..., 0.3961, 0.3725, 0.4118],
           [0.5059, 0.5216, 0.5373,  ..., 0.4196, 0.4275, 0.5059],
           [0.5059, 0.5137, 0.5216,  ..., 0.5059, 0.5294, 0.5922]],
 
          [[0.3569, 0.3961, 0.4275,  ..., 0.6157, 0.6706, 0.6784],
           [0.3333, 0.3569, 0.3804,  ..., 0.6627, 0.7255, 0.7647],
           [0.3333, 0.3333, 0.3412,  ..., 0.6863, 0.7020, 0.8118],
           ...,
           [0.3255, 0.3255, 0.3333,  ..., 0.2863, 0.2863, 0.3333],
           [0.3255, 0.3333, 0.3412,  ..., 0.3333, 0.3725, 0.4510],
           [0.3176, 0.3333, 0.3333,  ..., 0.4353, 0.4824, 0.5451]],
 
          [[0.3333, 0.3961, 0.4588,  ..., 0.6627, 0.7098, 0.7255],
           [0.3255, 0.3569, 0.4118,  ..., 0.7098, 0.7569, 0.8039],
           [0.3176, 0.33

In [20]:
next(iter(dataloader_B))

[tensor([[[[0.7725, 0.7490, 0.7882,  ..., 0.7961, 0.7569, 0.7176],
           [0.7647, 0.7725, 0.8118,  ..., 0.7569, 0.7333, 0.7098],
           [0.7569, 0.7961, 0.8431,  ..., 0.7333, 0.7255, 0.7255],
           ...,
           [0.8588, 0.8510, 0.8431,  ..., 0.6706, 0.6627, 0.6627],
           [0.8353, 0.8431, 0.8275,  ..., 0.6706, 0.6471, 0.6157],
           [0.8431, 0.8510, 0.8510,  ..., 0.6627, 0.6471, 0.6235]],
 
          [[0.6863, 0.6941, 0.7647,  ..., 0.7882, 0.7490, 0.7098],
           [0.6706, 0.7176, 0.7882,  ..., 0.7490, 0.7255, 0.7098],
           [0.6627, 0.7412, 0.8118,  ..., 0.7255, 0.7255, 0.7176],
           ...,
           [0.8510, 0.8431, 0.8353,  ..., 0.6627, 0.6471, 0.6471],
           [0.8275, 0.8353, 0.8196,  ..., 0.6549, 0.6392, 0.6078],
           [0.8275, 0.8431, 0.8431,  ..., 0.6471, 0.6314, 0.6157]],
 
          [[0.5529, 0.6392, 0.7725,  ..., 0.8275, 0.7882, 0.7412],
           [0.5059, 0.6549, 0.7725,  ..., 0.7882, 0.7647, 0.7490],
           [0.4824, 0.67

In [21]:
# Initialize models
G_AB = Generator().cuda()
G_BA = Generator().cuda()
D_A = Discriminator().cuda()
D_B = Discriminator().cuda()

# Loss functions
criterion_GAN = nn.MSELoss()  # Adversarial loss
criterion_cycle = nn.L1Loss()  # Cycle-consistency loss
criterion_identity = nn.L1Loss()

# Optimizers
optimizer_G = optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))


# Training Loop
for epoch in range(50):
    for i, (img_A, img_B) in enumerate(zip(dataloader_A, dataloader_B)):
        img_A, img_B = img_A[0].cuda(), img_B[0].cuda()
        
        # Train Generators
        optimizer_G.zero_grad()
        fake_B = G_AB(img_A)
        fake_A = G_BA(img_B)
        loss_GAN_AB = criterion_GAN(D_B(fake_B), torch.ones_like(D_B(fake_B)))
        loss_GAN_BA = criterion_GAN(D_A(fake_A), torch.ones_like(D_A(fake_A)))
        loss_cycle_A = criterion_cycle(G_BA(fake_B), img_A)
        loss_cycle_B = criterion_cycle(G_AB(fake_A), img_B)
        loss_identity_A = criterion_identity(G_BA(img_A), img_A)
        loss_identity_B = criterion_identity(G_AB(img_B), img_B)
        loss_G = loss_GAN_AB + loss_GAN_BA + 10 * (loss_cycle_A + loss_cycle_B) + 5 * (loss_identity_A + loss_identity_B)
        loss_G.backward()
        optimizer_G.step()

        # Train Discriminators
        optimizer_D_A.zero_grad()
        loss_D_A = criterion_GAN(D_A(img_A), torch.ones_like(D_A(img_A))) + criterion_GAN(D_A(fake_A.detach()), torch.zeros_like(D_A(fake_A)))
        loss_D_A.backward()
        optimizer_D_A.step()

        optimizer_D_B.zero_grad()
        loss_D_B = criterion_GAN(D_B(img_B), torch.ones_like(D_B(img_B))) + criterion_GAN(D_B(fake_B.detach()), torch.zeros_like(D_B(fake_B)))
        loss_D_B.backward()
        optimizer_D_B.step()

    print(f"Epoch {epoch}: Loss_G {loss_G.item()}, Loss_D_A {loss_D_A.item()}, Loss_D_B {loss_D_B.item()}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 15.69 GiB of which 205.56 MiB is free. Process 118026 has 2.91 GiB memory in use. Process 117896 has 2.38 GiB memory in use. Process 140201 has 2.79 GiB memory in use. Process 264984 has 2.30 GiB memory in use. Process 266371 has 2.30 GiB memory in use. Including non-PyTorch memory, this process has 1.84 GiB memory in use. Of the allocated memory 1.65 GiB is allocated by PyTorch, and 6.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)